In [2]:
import os, cv2, random, shutil, time
import numpy as np
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor

DATA_PATH = "URPC2020"
OUTPUT_PATH = "processed_dataset"

def clean_label(src, dest):
    if not os.path.exists(src):
        return False
    valid = []
    with open(src) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                valid.append(line.strip())
    if not valid:
        return False
    with open(dest, "w") as f:
        f.write("\n".join(valid))
    return True

def apply_underwater(img):
    img = cv2.convertScaleAbs(img, alpha=0.6, beta=-20)
    noise = np.random.randint(0,10,img.shape,dtype='uint8')
    img = cv2.add(img, noise)
    b,g,r = cv2.split(img)
    b = cv2.add(b,15)
    g = cv2.add(g,8)
    return cv2.merge((b,g,r))

def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l,a,b = cv2.split(lab)
    cl = cv2.createCLAHE(2.0,(8,8)).apply(l)
    return cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2BGR)

def enhance(img):
    return apply_clahe(cv2.convertScaleAbs(img, alpha=1.2, beta=10))

def process_file(args):
    img_name, img_src, lbl_src, img_dest, lbl_dest, mode = args

    img_path = os.path.join(img_src, img_name)
    lbl_path = os.path.join(lbl_src, img_name.rsplit('.',1)[0]+".txt")

    if not os.path.exists(lbl_path):
        return

    img = cv2.imread(img_path)
    if img is None:
        return

    if mode == "original":
        out = img
    elif mode == "noisy":
        out = apply_underwater(img)
    else:
        out = enhance(apply_underwater(img))

    dest_img = os.path.join(img_dest, img_name)
    dest_lbl = os.path.join(lbl_dest, os.path.basename(lbl_path))

    if not clean_label(lbl_path, dest_lbl):
        return

    cv2.imwrite(dest_img, out, [cv2.IMWRITE_JPEG_QUALITY,85])

def create_dataset(mode):
    print(f"\nCreating {mode}")

    for split in ["train","valid","test"]:
        start = time.time()

        img_src = f"{DATA_PATH}/{split}/images"
        lbl_src = f"{DATA_PATH}/{split}/labels"

        img_dest = f"{OUTPUT_PATH}/{mode}/{split}/images"
        lbl_dest = f"{OUTPUT_PATH}/{mode}/{split}/labels"

        os.makedirs(img_dest, exist_ok=True)
        os.makedirs(lbl_dest, exist_ok=True)

        files = os.listdir(img_src)
        tasks = [(f, img_src, lbl_src, img_dest, lbl_dest, mode) for f in files]

        with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
            list(tqdm(executor.map(process_file, tasks), total=len(tasks)))

        print(f"{mode}-{split} done in {round(time.time()-start,2)}s")

def merge_mixed(limit=2000):
    print("\nCreating mixed dataset")

    for split in ["train","valid","test"]:
        img_dest = f"{OUTPUT_PATH}/mixed/{split}/images"
        lbl_dest = f"{OUTPUT_PATH}/mixed/{split}/labels"

        os.makedirs(img_dest, exist_ok=True)
        os.makedirs(lbl_dest, exist_ok=True)

        for mode in ["original","noisy","enhanced"]:
            img_src = f"{OUTPUT_PATH}/{mode}/{split}/images"
            lbl_src = f"{OUTPUT_PATH}/{mode}/{split}/labels"

            files = os.listdir(img_src)
            if len(files) > limit:
                files = random.sample(files, limit)

            count = 0
            for file in files:
                label = file.rsplit('.',1)[0]+".txt"

                src_img = os.path.join(img_src,file)
                src_lbl = os.path.join(lbl_src,label)

                if not os.path.exists(src_lbl):
                    continue

                shutil.copy(src_img, os.path.join(img_dest,f"{mode}_{file}"))
                shutil.copy(src_lbl, os.path.join(lbl_dest,f"{mode}_{label}"))
                count += 1

            print(f"{mode}-{split}: {count} added")

if __name__ == "__main__":
    os.makedirs(OUTPUT_PATH, exist_ok=True)

    create_dataset("original")
    create_dataset("noisy")
    create_dataset("enhanced")
    merge_mixed(2000)

    print("\nDone. Dataset ready in:", OUTPUT_PATH)

ModuleNotFoundError: No module named 'cv2'